# ConceptNet-100k: независимые полные запуски Wishart при k=4 и k=5

Этот блокнот **дважды читает исходный ConceptNet**, строит граф до 100 000 вершин и выполняет полный frequency scan на каждом завершённом уровне. У каждого k — отдельные словарь, MDL-отбор, иерархия и контрольные точки. Обнаружение новых типов по-прежнему начинается с выборки кандидатов (5000 центров), затем известные типы проверяются полным обходом.

После каждого завершённого запуска сохраняется **факторизованный словарь** (одна точная внутренняя форма + отдельные внешние порты каждого типа), таблица внешних рёбер с весами и динамические показатели по макроузлам и уровням. Восстановление каждого прототипа проверяется точно; это пока **не полный бинарный кодек исходного графа**: его матрицы и контрольные точки сохраняются отдельно.

Выберите в Colab среду с GPU и достаточной RAM. При прерывании смените RUN_MODE на RESUME; параметры и Git-ревизию менять нельзя. До запуска укажите реальный путь к исходному TSV в DATASET_DRIVE. Подробнее: docs/wishart_100k_k4_k5_shared_shapes.md.

In [ ]:
# 1. Все редактируемые параметры.
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, platform
from datetime import datetime, timezone

DRIVE_ROOT = Path("/content/drive/MyDrive/SemanticMap/colab/wishart")
DATASET_DRIVE = "datasets/conceptnet_en_100k.tsv"  # ИЗМЕНИТЕ на путь к вашему сырому TSV относительно DRIVE_ROOT.
K_VALUES = (4, 5)
RUN_MODE = "NEW"  # NEW | RESUME. Для прерванного запуска переключите на RESUME.
RUN_PREFIX = "wishart-cn100k"
CPU_WORKERS = 4  # фиксируем между сессиями; CLI ограничит число доступными CPU
REPO_URL = "https://github.com/SemanticMap/semgraphex.git"
BRANCH = "feature/wishart-100k-k4-k5-shared-shapes"
CODE_COMMIT = "2dc7cdd452283afa1b83ed178fc09238659452e8"
REPO_DIR = Path("/content/semgraphex-k4-k5")
SCRATCH = Path("/content/semmap-wishart-k4-k5")
ANALYSIS_NAME = "shared-shapes-v1"
if RUN_MODE not in {"NEW", "RESUME"}:
    raise ValueError("RUN_MODE must be NEW or RESUME")
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"
SCRATCH.mkdir(parents=True, exist_ok=True)
print({"python": sys.version.split()[0], "cpus": os.cpu_count(),
       "workers": CPU_WORKERS, "free_local_gb": round(shutil.disk_usage(SCRATCH).free / 1e9, 1),
       "k_values": K_VALUES, "run_mode": RUN_MODE})

In [ ]:
# 2. Монтируем Drive, проверяем наличие СЫРОГО источника и место под копию.
from google.colab import drive
drive.mount("/content/drive")
dataset_path = DRIVE_ROOT / DATASET_DRIVE
if not dataset_path.is_file():
    raise FileNotFoundError(f"Укажите реальный DATASET_DRIVE (исходный ConceptNet TSV): {dataset_path}")
dataset_size = dataset_path.stat().st_size
free = shutil.disk_usage(SCRATCH).free
recommended = max(dataset_size * 3, dataset_size + 2 * 1024**3)
if free < recommended:
    raise RuntimeError(f"Недостаточно места в /content: свободно {free/1e9:.1f} ГБ, нужно около {recommended/1e9:.1f} ГБ")
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    gpu = None
print({"dataset": str(dataset_path), "source_gb": round(dataset_size/1e9, 2),
       "gpu": gpu, "cuda_requested": "auto", "gpu_note": "CUDA используется для typed-WL kNN; VF2 и диагностика остаются на CPU"})
if gpu is None:
    print("ПРЕДУПРЕЖДЕНИЕ: GPU не найден. Запуск возможен на CPU, но будет медленнее.")

In [ ]:
# 3. Фиксированная ревизия кода, явная установка пакета и проверка импортов.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "30", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", CODE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
assert actual == CODE_COMMIT, (actual, CODE_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "pytest>=7.4"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
from semmap_haken.wishart_colab import sync_tree
from semmap_haken.wishart_shared_shapes import export_run
from semmap_haken.wishart_config import load_wishart_options, load_dictionary_options
import yaml
print({"pinned_commit": actual, "module": export_run.__module__})

In [ ]:
# 4. Создаём два независимых YAML и сохраняем их на Drive для строгого RESUME.
base = yaml.safe_load((REPO_DIR / "configs/wishart_conceptnet_100k_shared_shapes.yaml").read_text(encoding="utf-8"))
assert base["dataset"]["max_nodes"] == 100000
assert base["dictionary"]["frequency_scan"] == "full"
assert base["dictionary"]["boundary_sensitive"] is True
assert base["wishart"]["metric"] == "typed_wl"
assert base["wishart"]["clustering_domain"] == "canonical_types"
CONFIG_DIR = DRIVE_ROOT / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIGS = {}
RUN_NAMES = {}
for k in K_VALUES:
    config = json.loads(json.dumps(base))
    config["wishart"]["k_neighbors"] = k
    config["colab"]["device"] = "auto"
    config["colab"]["cpu_workers"] = CPU_WORKERS
    config["colab"]["checkpoint_every_levels"] = 1
    run_name = f"{RUN_PREFIX}-k{k}-shared-v1"
    config_path = CONFIG_DIR / f"{run_name}.yaml"
    rendered = yaml.safe_dump(config, sort_keys=True, allow_unicode=True)
    if config_path.exists():
        if config_path.read_text(encoding="utf-8") != rendered:
            raise ValueError(f"Конфиг на Drive отличается; используйте другое имя запуска: {config_path}")
    else:
        config_path.write_text(rendered, encoding="utf-8")
    CONFIGS[k] = config_path
    RUN_NAMES[k] = run_name
    assert load_wishart_options(config_path).k_neighbors == k
    assert load_dictionary_options(config_path).frequency_scan == "full"
print({"configs": {k: str(v) for k, v in CONFIGS.items()}, "runs": RUN_NAMES})

In [ ]:
# 5. Быстрые регрессионные тесты факторизации до дорогого полного запуска.
subprocess.run([sys.executable, "-m", "pytest", "-q",
                str(REPO_DIR / "tests/test_semmap_haken_shared_shapes.py")],
               cwd=REPO_DIR, check=True)
print("Проверены: точное восстановление прототипа, VF2-перестановка и внешние рёбра.")

In [ ]:
# 6. Два ПОЛНЫХ запуска: каждый читает исходный TSV и заново проходит 100k-граф.
# Checkpoints исходного runner синхронизируются в Drive после каждого завершённого уровня.
# После сбоя запустите снова с RUN_MODE="RESUME" и той же конфигурацией.
results = {}
for k in K_VALUES:
    name = RUN_NAMES[k]
    drive_run = DRIVE_ROOT / "runs" / name
    drive_analysis = DRIVE_ROOT / "analysis" / name / ANALYSIS_NAME
    local_run = SCRATCH / "runs" / name
    local_analysis = SCRATCH / "analysis" / name / ANALYSIS_NAME
    if (drive_run / "COMPLETED").is_file() and (drive_analysis / "COMPLETED").is_file():
        if RUN_MODE != "RESUME":
            raise RuntimeError(f"{name} уже завершён. Выберите RUN_MODE='RESUME' для просмотра результатов.")
        print("Уже завершён:", name)
    else:
        cmd = [sys.executable, "-m", "semmap_haken.wishart_colab_cli",
               "--config-drive", str(CONFIGS[k].relative_to(DRIVE_ROOT)),
               "--dataset-drive", DATASET_DRIVE,
               "--drive-root", str(DRIVE_ROOT),
               "--scratch-root", str(SCRATCH),
               "--run-name", name,
               "--device", "auto",
               "--cpu-workers", str(CPU_WORKERS),
               "--blas-threads", "1",
               "--keep-scratch"]
        if RUN_MODE == "RESUME":
            cmd.append("--resume")
        subprocess.run(cmd, cwd=REPO_DIR, check=True)
        assert (local_run / "COMPLETED").is_file(), f"Run {name} не завершился"
        if not (drive_analysis / "COMPLETED").is_file():
            report = export_run(local_run, local_analysis)
            assert report["dictionary"]["prototype_roundtrip_exact"]
            sync_tree(local_analysis, drive_analysis, compare_mode="size_mtime", final=True)
            assert (drive_analysis / "COMPLETED").is_file(), "Экспорт не синхронизирован"
            print("Факторизованный словарь и динамика:", drive_analysis)
        # Большой локальный run не нужен после подтверждённой синхронизации.
        shutil.rmtree(local_run, ignore_errors=True)
    results[k] = {"run": drive_run, "analysis": drive_analysis}
print("Оба независимых k завершены и сохранены.")

In [ ]:
# 7. Сопоставимые результаты. Не называем оценку словаря размером полного кодека.
comparison = {}
for k, paths in results.items():
    hierarchy = json.loads((paths["run"] / "hierarchy.json").read_text(encoding="utf-8"))
    report = json.loads((paths["analysis"] / "report.json").read_text(encoding="utf-8"))
    assert report["dictionary"]["prototype_roundtrip_exact"]
    comparison[k] = {
        "initial_nodes": hierarchy["initial_nodes"],
        "final_nodes": hierarchy["final_nodes"],
        "stop_reason": hierarchy["stop_reason"],
        "transitions": len(hierarchy["transitions"]),
        "exact_types": report["dictionary"]["exact_types"],
        "shared_internal_shapes": report["dictionary"]["shared_internal_shapes"],
        "dictionary_jsonl_saved_bytes": report["dictionary"]["jsonl_saved_bytes"],
        "dictionary_gzip_saved_bytes": report["dictionary"]["gzip_saved_bytes"],
        "accepted_figures": sum(x["figures"] for x in report["transitions"]),
        "external_relation_entries": sum(x["external_relation_entries"] for x in report["transitions"]),
    }
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print("100000 — ограничение на число узлов; фактическое число указано в initial_nodes.")

In [ ]:
# 8. Итоговый манифест. Маркер завершения — после всех результатов.
manifest_dir = DRIVE_ROOT / "analysis" / f"{RUN_PREFIX}-k4-k5-shared-v1"
manifest_dir.mkdir(parents=True, exist_ok=True)
(manifest_dir / "COMPLETED").unlink(missing_ok=True)
manifest = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": CODE_COMMIT,
    "run_mode": RUN_MODE,
    "dataset_drive": DATASET_DRIVE,
    "dataset_bytes": dataset_size,
    "k_values": list(K_VALUES),
    "comparison": comparison,
    "configs": {str(k): str(CONFIGS[k]) for k in K_VALUES},
    "runs": {str(k): str(results[k]["run"]) for k in K_VALUES},
    "analyses": {str(k): str(results[k]["analysis"]) for k in K_VALUES},
    "scope": "full independent Wishart runs; exact prototype factorization and per-figure boundary edges; not a full standalone graph codec",
}
(manifest_dir / "comparison.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(manifest_dir / "COMPLETED").write_text("complete\n", encoding="utf-8")
print("Сохранено:", manifest_dir / "comparison.json")

## Интерпретация

В каждом каталоге analysis/<run-name>/shared-shapes-v1/ есть report.json, dictionary_factorized/, а для каждого перехода — figure_dynamics.jsonl, graph_metrics.json и external_connections.jsonl.gz. MFPT, спектральный разрыв, пороги, средняя степень, кластеризация и межкластерные расстояния относятся к **графу уровня**, а не к каждому отдельному подграфу. Betweenness и координаты медленных мод относятся к **макроузлу целевого уровня**; поток, масса до сжатия и вероятности выхода — к его объединяемым вершинам на исходном уровне. Знаки собственных векторов между запусками могут различаться.

Исходные CSR-матрицы и контрольные точки остаются в runs/. Выгода по размеру словаря не равна выигрышу полного кодека.